# The WSP Heuristic

Here we calculate the WSPD for a given problem and then check whether the number of edges crossing the each pair of clusters is the correct number given the number of exits out of the cluster as per the proved theorems. 

In [9]:
from pathlib import Path
from time import perf_counter

import tsplib95
import numpy as np

from wsp import ds, tsp

import sys

TREE_TYPE = ds.PKPRQuadTree
SIZE_THRESHOLD = 250
TSP_LIB_PATH = Path("TSPLIB")
S_FACTOR = 1.01
#ax = np.array([None, None])
#sys.setrecursionlimit(500_000)

In [3]:
#all_problems : dict[str, tsp.TravellingSalesmanProblem[TREE_TYPE]] = {}

#for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
#    if file.suffix != ".tsp" or not file.is_file():
#        continue
#    problem = tsplib95.load(f"{file.absolute()}")
#    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
#        continue
#    if problem.dimension > SIZE_THRESHOLD: # Skip large TSPs
#        continue

#    points = [ds.Point(*problem.node_coords[i]) for i in problem.get_nodes()]
#    ts_problem = tsp.TravellingSalesmanProblem[TREE_TYPE](TREE_TYPE, points, ax, s=S_FACTOR) 
    
#    all_problems[problem.name] = ts_problem
#    print(f"Added {problem.name}")

#print("Found", len(all_problems), "euclidean TSPs")

In [4]:
#for name, problem in all_problems.items():
#    tour, _, _ = problem.nnn_path
#    badCount = 0
#    for (A, B), _ in problem.wspd:
#        pointsA = set(A.covered_points)
#        pointsB = set(B.covered_points)
#        if not problem.wsp_heuristic_good(tour, pointsA, pointsB):
#            badCount += 1
#    if badCount > 0:
#        print(f"Problem {name} has {badCount} bad pairs")

## Faster Runner using C libs

In [5]:
import elkai
from wspd import build_wspd, point as wsp_point # uses diam based sep factor

In [ ]:
all_problems : list[tsplib95.models.StandardProblem] = []

for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
    if file.suffix != ".tsp" or not file.is_file():
        continue
    problem = tsplib95.load(f"{file.absolute()}")
    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
        continue
    if problem.dimension > SIZE_THRESHOLD: # Skip large TSPs
        continue
    all_problems.append(problem)

In [ ]:
def wsp_heuristic_good(path: np.ndarray, A: set[int], B: set[int]) -> bool:
        """A heuristic to check if a path is good based on the WSPs, checks if there are at least 2 connections between A and B"""
        assert len(A) > 0 and len(B) > 0, "Sets must be non-empty"
        exit_A, exit_B = 0, 0
        combo = A | B
        biconn_count = 0
        for i in range(len(path) - 1):
            if (path[i] in A and path[i + 1] not in combo) or (path[i] not in combo and path[i + 1] in A):
                exit_A += 1
            elif (path[i] in B and path[i + 1] not in combo) or (path[i] not in combo and path[i + 1] in B):
                exit_B += 1
            elif (path[i] in A and path[i + 1] in B) or (path[i] in B and path[i + 1] in A):
                biconn_count += 1

        # single ham path case
        #if exit_A == 1 and exit_B == 1: # covered in the multi case
        #    return biconn_count == 1
        #if (exit_A == 2 and exit_B == 0) or (exit_A == 0 and exit_B == 2): # covered in the multi case
        #    return biconn_count == 2
        # mult ham path case
        if exit_A == 0 or exit_B == 0:
            return biconn_count == 2
        elif exit_A % 2 == 0:
            return biconn_count == 0
        else:
            return biconn_count == 1

In [ ]:
now = perf_counter()
for prob in all_problems:
    print(f"Checking {prob.name}...")
    points = [wsp_point(prob.node_coords[i]) for i in prob.get_nodes()]
    wpsd: list[tuple[list[int], list[int]]] = []
    wspd = build_wspd(len(points), 2, S_FACTOR, points)

    # pruning problems which cannot be bad
    wspd = [(A, B) for A, B in wspd if len(A) > 1 and len(B) > 1]

    lkh_instance = elkai.Coordinates2D({
        i-1: prob.node_coords[i] for i in prob.get_nodes()
    })
    tour = np.array(lkh_instance.solve_tsp(), dtype=np.int16)

    for A, B in wspd:
        pointsA = set(A)
        pointsB = set(B)
        if not wsp_heuristic_good(tour, pointsA, pointsB):
            print(f"Problem {prob.name} has a bad pair: {A}, {B}")

print("Checked all problems in", perf_counter() - now, "seconds")

Checking berlin52...
Checking bier127...
Checking ch130...
Checking ch150...
Checking d198...
Checking eil101...
Checking eil51...
Checking eil76...
Checking kroA100...
Checking kroA150...
Checking kroA200...
Checking kroB100...
Checking kroB150...
Checking kroB200...
Checking kroC100...
Checking kroD100...
Checking kroE100...
Checking lin105...
Checking pr107...
Checking pr124...
Checking pr136...
Checking pr144...
Checking pr152...
Checking pr226...
Checking pr76...
Checking rat195...
Checking rat99...
Checking rd100...
Checking st70...
Checking ts225...
Checking tsp225...
Checking u159...
Checked all problems in 30.529651667020516 seconds
